In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

In [3]:
df = sns.load_dataset('iris')

In [4]:
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [5]:

encoder = LabelEncoder()
df['species'] = encoder.fit_transform(df['species'])

In [6]:
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [7]:
X = df.iloc[:, 0:2]
y = df['species']

X.shape

(150, 2)

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

In [9]:
clf1 = LogisticRegression()
clf2 = RandomForestClassifier()
clf3 = KNeighborsClassifier()

estimators = [('lr',clf1),('rf',clf2),('knn',clf3)]

In [10]:
for estimator in estimators:
    scores = cross_val_score(estimator[1],X,y, cv=10, scoring='accuracy')
    print(f"{estimator[0].upper():>5} Accuracy: {np.round(np.mean(scores), 2)}")


   LR Accuracy: 0.81
   RF Accuracy: 0.71
  KNN Accuracy: 0.76


In [11]:
from sklearn.ensemble import VotingClassifier

vc_hard = VotingClassifier(estimators=estimators, voting="hard")

scores_hard = cross_val_score(vc_hard,X,y, cv=10, scoring='accuracy')
print(f"Hard Voting Accuracy: {np.round(np.mean(scores_hard), 2)}")

Hard Voting Accuracy: 0.77


In [12]:
vc_soft = VotingClassifier(estimators=estimators, voting='soft')

scores_soft = cross_val_score(vc_soft, X, y, cv=10, scoring='accuracy')
print(f"Soft Voting Accuracy: {np.round(np.mean(scores_soft), 2)}")

Soft Voting Accuracy: 0.77


In [13]:
# Test all weight combinations from 1 to 3 for each of the 3 models
print("--- Weighted Voting: All Combinations ---")
best_score = 0
best_weights = None

for i in range(1, 4):       # Weight for LR
    for j in range(1, 4):   # Weight for RF
        for k in range(1, 4):  # Weight for KNN
            vc_weighted = VotingClassifier(
                estimators=estimators, 
                voting='soft', 
                weights=[i, j, k]
            )
            scores = cross_val_score(vc_weighted, X, y, cv=10, scoring='accuracy')
            avg_score = np.round(np.mean(scores), 2)
            
            # Track the best combination
            if avg_score > best_score:
                best_score = avg_score
                best_weights = (i, j, k)
            
            print(f"LR={i}, RF={j}, KNN={k} --> Accuracy: {avg_score}")

print(f"\nBest Weights: LR={best_weights[0]}, RF={best_weights[1]}, KNN={best_weights[2]}")
print(f"Best Accuracy: {best_score}")

--- Weighted Voting: All Combinations ---
LR=1, RF=1, KNN=1 --> Accuracy: 0.77
LR=1, RF=1, KNN=2 --> Accuracy: 0.77
LR=1, RF=1, KNN=3 --> Accuracy: 0.77
LR=1, RF=2, KNN=1 --> Accuracy: 0.75
LR=1, RF=2, KNN=2 --> Accuracy: 0.76
LR=1, RF=2, KNN=3 --> Accuracy: 0.76
LR=1, RF=3, KNN=1 --> Accuracy: 0.73
LR=1, RF=3, KNN=2 --> Accuracy: 0.75
LR=1, RF=3, KNN=3 --> Accuracy: 0.77
LR=2, RF=1, KNN=1 --> Accuracy: 0.77
LR=2, RF=1, KNN=2 --> Accuracy: 0.77
LR=2, RF=1, KNN=3 --> Accuracy: 0.77
LR=2, RF=2, KNN=1 --> Accuracy: 0.77
LR=2, RF=2, KNN=2 --> Accuracy: 0.77
LR=2, RF=2, KNN=3 --> Accuracy: 0.77
LR=2, RF=3, KNN=1 --> Accuracy: 0.75
LR=2, RF=3, KNN=2 --> Accuracy: 0.77
LR=2, RF=3, KNN=3 --> Accuracy: 0.76
LR=3, RF=1, KNN=1 --> Accuracy: 0.81
LR=3, RF=1, KNN=2 --> Accuracy: 0.78
LR=3, RF=1, KNN=3 --> Accuracy: 0.79
LR=3, RF=2, KNN=1 --> Accuracy: 0.78
LR=3, RF=2, KNN=2 --> Accuracy: 0.78
LR=3, RF=2, KNN=3 --> Accuracy: 0.77
LR=3, RF=3, KNN=1 --> Accuracy: 0.77
LR=3, RF=3, KNN=2 --> Accuracy: 0

In [15]:
from sklearn.svm import SVC
from sklearn.datasets import make_classification

X_svm, y_svm = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    random_state=2
)

svm1 = SVC(probability=True, kernel='poly', degree=1)
svm2 = SVC(probability=True, kernel='poly', degree=2)
svm3 = SVC(probability=True, kernel='poly', degree=3)
svm4 = SVC(probability=True, kernel='poly', degree=4)
svm5 = SVC(probability=True, kernel='poly', degree=5)

svm_estimators = [
    ('svm1', svm1), ('svm2', svm2), ('svm3', svm3), 
    ('svm4', svm4), ('svm5', svm5)
]

# Check individual scores first
for est in svm_estimators:
    scores = cross_val_score(est[1], X_svm, y_svm, cv=10, scoring='accuracy')
    print(f"{est[0].upper()} (degree={est[1].degree}) Accuracy: {np.round(np.mean(scores), 2)}")

# combine them using Soft Voting
print("\n--- Soft Voting (All 5 SVMs Combined) ---")
vc_svm = VotingClassifier(estimators=svm_estimators, voting='soft')
scores_combined = cross_val_score(vc_svm, X_svm, y_svm, cv=10, scoring='accuracy')
print(f"Combined SVM Voting Accuracy: {np.round(np.mean(scores_combined), 2)}")


c:\Users\husna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\husna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\husna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\husna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\

SVM1 (degree=1) Accuracy: 0.85


c:\Users\husna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\husna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\husna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\husna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\

SVM2 (degree=2) Accuracy: 0.85


c:\Users\husna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\husna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\husna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\husna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\

...

SVM4 (degree=4) Accuracy: 0.81


c:\Users\husna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\husna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\husna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\husna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\

SVM5 (degree=5) Accuracy: 0.86

--- Soft Voting (All 5 SVMs Combined) ---


c:\Users\husna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\husna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\husna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\husna\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\

Combined SVM Voting Accuracy: 0.93
